## 📦 Install Dependencies

In [ ]:
!pip install plotly scipy numpy pandas ipywidgets

## 🧪 Generate Synthetic BCI Sessions

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

np.random.seed(42)

NUM_SESSIONS = 5
DURATION = 600  # 10 minutes at 1Hz
time_axis = np.arange(DURATION)

def sigmoid(x, center, steepness=0.1):
    """Smooth sigmoid transition."""
    return 1 / (1 + np.exp(-steepness * (x - center)))

def generate_focus_signal(session_idx):
    """Generate a realistic BCI focus signal with smooth phase transitions."""
    t = np.arange(DURATION, dtype=float)
    base_offset = np.random.uniform(-5, 5)
    noise_amp = np.random.uniform(2, 5)

    # Ramp-up phase (0-60s): focus rises from ~30 to ~70
    ramp_up = 30 + 40 * sigmoid(t, center=30, steepness=0.08)

    # Plateau phase (60-300s): focus stays around 65-80
    plateau = 72 + base_offset

    # Distraction dip (300-360s): focus drops to ~40-50
    dip_depth = np.random.uniform(20, 30)
    dip = -dip_depth * np.exp(-0.5 * ((t - 330) / 20) ** 2)

    # Recovery (360-500s): focus returns to ~65-75
    recovery = dip_depth * sigmoid(t, center=400, steepness=0.05)

    # Combine phases using smooth blending
    blend_to_plateau = sigmoid(t, center=60, steepness=0.1)
    signal = ramp_up * (1 - blend_to_plateau) + plateau * blend_to_plateau

    # Add dip and recovery
    signal = signal + dip + recovery * sigmoid(t, center=360, steepness=0.1)

    # Final plateau adjustment (500-600s): stable around 70-75
    final_blend = sigmoid(t, center=500, steepness=0.08)
    final_level = 72 + base_offset * 0.5
    signal = signal * (1 - final_blend * 0.3) + final_level * (final_blend * 0.3)

    # Add Gaussian noise
    noise = np.random.normal(0, noise_amp, DURATION)
    signal = signal + noise

    # Clip to valid range
    signal = np.clip(signal, 0, 100)
    return signal

# Generate sessions
sessions = {}
for i in range(NUM_SESSIONS):
    session_id = f"Session {i + 1}"
    sessions[session_id] = generate_focus_signal(i)

# Create combined DataFrame
frames = []
for sid, data in sessions.items():
    df_sess = pd.DataFrame({
        "time_s": time_axis,
        "focus_score": data,
        "session_id": sid
    })
    frames.append(df_sess)

df = pd.concat(frames, ignore_index=True)

# Summary statistics
print("=== BCI Focus Data Summary ===")
print(f"Sessions: {NUM_SESSIONS}")
print(f"Duration: {DURATION}s per session")
print(f"Total data points: {len(df)}")
print()
print(df.groupby("session_id")["focus_score"].describe().round(2))

## 📈 Multi-Session Time-Series Overlay

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Individual session lines
for sid, data in sessions.items():
    fig.add_trace(go.Scatter(
        x=time_axis, y=data,
        mode="lines",
        name=sid,
        opacity=0.4,
        line=dict(width=1)
    ))

# Bold average line
mean_signal = np.mean([sessions[s] for s in sessions], axis=0)
fig.add_trace(go.Scatter(
    x=time_axis, y=mean_signal,
    mode="lines",
    name="Mean",
    opacity=1.0,
    line=dict(width=3, color="white")
))

fig.update_layout(
    template="plotly_dark",
    title="BCI Focus Scores — Multi-Session Overlay",
    xaxis_title="Time (seconds)",
    yaxis_title="Focus Score"
)

fig.show()

## 🔍 Rolling Average + Peak Detection

In [ ]:
from scipy.signal import find_peaks

# Rolling average on mean signal
window = 30
rolling_avg = pd.Series(mean_signal).rolling(window=window, center=True).mean().values

# Detect peaks and troughs
peaks, peak_props = find_peaks(rolling_avg[~np.isnan(rolling_avg)], prominence=5)
troughs, trough_props = find_peaks(-rolling_avg[~np.isnan(rolling_avg)], prominence=5)

# Adjust indices for NaN offset from rolling window
valid_mask = ~np.isnan(rolling_avg)
valid_indices = np.where(valid_mask)[0]
peak_times = valid_indices[peaks]
trough_times = valid_indices[troughs]

fig = go.Figure()

# Raw mean signal
fig.add_trace(go.Scatter(
    x=time_axis, y=mean_signal,
    mode="lines",
    name="Raw Mean Signal",
    line=dict(width=1, color="gray"),
    opacity=0.5
))

# Rolling average
fig.add_trace(go.Scatter(
    x=time_axis, y=rolling_avg,
    mode="lines",
    name=f"Rolling Average (w={window})",
    line=dict(width=3, color="cyan")
))

# Peaks
fig.add_trace(go.Scatter(
    x=peak_times, y=rolling_avg[peak_times],
    mode="markers",
    name="Peaks",
    marker=dict(symbol="triangle-up", size=12, color="red")
))

# Troughs
fig.add_trace(go.Scatter(
    x=trough_times, y=rolling_avg[trough_times],
    mode="markers",
    name="Troughs",
    marker=dict(symbol="triangle-down", size=12, color="blue")
))

fig.update_layout(
    template="plotly_dark",
    title="Focus Signal — Rolling Average with Peak Detection",
    xaxis_title="Time (seconds)",
    yaxis_title="Focus Score"
)

fig.show()

## 🏔️ 3D Focus Surface

In [ ]:
# Build 2D array: rows=sessions, cols=time
z_data = np.array([sessions[f"Session {i+1}"] for i in range(NUM_SESSIONS)])

fig = go.Figure(data=[go.Surface(
    x=time_axis,
    y=np.arange(1, NUM_SESSIONS + 1),
    z=z_data,
    colorscale="Viridis",
    colorbar=dict(title="Focus Score")
)])

fig.update_layout(
    template="plotly_dark",
    title="Focus Landscape — 3D Surface (Time × Session × Focus)",
    scene=dict(
        xaxis_title="Time (seconds)",
        yaxis_title="Session",
        zaxis_title="Focus Score",
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
    )
)

fig.show()

## 🎻 Focus Distribution — Violin Plots

In [ ]:
fig = go.Figure()

for sid in sessions:
    fig.add_trace(go.Violin(
        y=sessions[sid],
        name=sid,
        box_visible=True,
        meanline_visible=True
    ))

fig.update_layout(
    template="plotly_dark",
    title="Focus Score Distribution by Session",
    yaxis_title="Focus Score",
    showlegend=False
)

fig.show()

## 🔗 Correlation: Focus vs Node Creation

In [ ]:
# Generate synthetic node creation events
# Higher probability of node creation during high-focus moments (focus > 70)
np.random.seed(42)

# For each time point in the mean signal, generate events
focus_prob = np.clip((mean_signal - 40) / 60, 0.05, 0.95)
node_events = np.random.binomial(1, focus_prob)

# Compute node creation rate in a sliding window
rate_window = 30
node_creation_rate = pd.Series(node_events).rolling(
    window=rate_window, center=True
).mean().values

# Remove NaN edges
valid = ~np.isnan(node_creation_rate)
x_valid = mean_signal[valid]
y_valid = node_creation_rate[valid]

# OLS regression
coeffs = np.polyfit(x_valid, y_valid, 1)
slope, intercept = coeffs
y_pred = slope * x_valid + intercept

# R² calculation
ss_res = np.sum((y_valid - y_pred) ** 2)
ss_tot = np.sum((y_valid - np.mean(y_valid)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

fig = go.Figure()

# Scatter
fig.add_trace(go.Scatter(
    x=x_valid, y=y_valid,
    mode="markers",
    name="Data Points",
    marker=dict(size=4, opacity=0.5, color="cyan")
))

# Regression line
x_sorted = np.sort(x_valid)
fig.add_trace(go.Scatter(
    x=x_sorted,
    y=slope * x_sorted + intercept,
    mode="lines",
    name="OLS Fit",
    line=dict(color="red", width=2)
))

fig.update_layout(
    template="plotly_dark",
    title="Focus Score vs Node Creation Rate",
    xaxis_title="Focus Score",
    yaxis_title="Node Creation Rate",
    annotations=[dict(
        x=0.05, y=0.95,
        xref="paper", yref="paper",
        text=f"R² = {r_squared:.4f}",
        showarrow=False,
        font=dict(size=14, color="white"),
        bgcolor="rgba(0,0,0,0.5)"
    )]
)

fig.show()

## 🎛️ Sandbox — BCI Session Explorer

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

print("🎛️ Sandbox — BCI Session Explorer")

@interact(
    smoothing_window=widgets.IntSlider(min=5, max=60, value=30, description="Smoothing:"),
    peak_threshold=widgets.IntSlider(min=1, max=20, value=5, step=1, description="Peak Thr:"),
    session=widgets.Dropdown(
        options=["All"] + [f"Session {i+1}" for i in range(NUM_SESSIONS)],
        value="All",
        description="Session:"
    )
)
def explore(smoothing_window, peak_threshold, session):
    # Select signal
    if session == "All":
        signal = mean_signal
        title_suffix = "All Sessions (Mean)"
    else:
        signal = sessions[session]
        title_suffix = session

    # Rolling average
    smooth = pd.Series(signal).rolling(
        window=smoothing_window, center=True
    ).mean().values

    # Peak detection
    valid_mask = ~np.isnan(smooth)
    valid_vals = smooth[valid_mask]
    valid_idx = np.where(valid_mask)[0]

    p, _ = find_peaks(valid_vals, prominence=peak_threshold)
    tr, _ = find_peaks(-valid_vals, prominence=peak_threshold)

    p_times = valid_idx[p]
    tr_times = valid_idx[tr]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=time_axis, y=signal,
        mode="lines", name="Raw Signal",
        line=dict(width=1, color="gray"), opacity=0.4
    ))

    fig.add_trace(go.Scatter(
        x=time_axis, y=smooth,
        mode="lines", name=f"Rolling Avg (w={smoothing_window})",
        line=dict(width=3, color="cyan")
    ))

    fig.add_trace(go.Scatter(
        x=p_times, y=smooth[p_times],
        mode="markers", name="Peaks",
        marker=dict(symbol="triangle-up", size=12, color="red")
    ))

    fig.add_trace(go.Scatter(
        x=tr_times, y=smooth[tr_times],
        mode="markers", name="Troughs",
        marker=dict(symbol="triangle-down", size=12, color="blue")
    ))

    fig.update_layout(
        template="plotly_dark",
        title=f"BCI Explorer — {title_suffix}",
        xaxis_title="Time (seconds)",
        yaxis_title="Focus Score"
    )

    fig.show()

## 💾 Export

In [ ]:
import os

os.makedirs("exports", exist_ok=True)

# Recreate 3D surface figure for export
z_export = np.array([sessions[f"Session {i+1}"] for i in range(NUM_SESSIONS)])

fig_export = go.Figure(data=[go.Surface(
    x=time_axis,
    y=np.arange(1, NUM_SESSIONS + 1),
    z=z_export,
    colorscale="Viridis",
    colorbar=dict(title="Focus Score")
)])

fig_export.update_layout(
    template="plotly_dark",
    title="Focus Landscape — 3D Surface (Time × Session × Focus)",
    scene=dict(
        xaxis_title="Time (seconds)",
        yaxis_title="Session",
        zaxis_title="Focus Score",
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
    )
)

fig_export.write_html("exports/bci_focus_analysis.html")
print("✅ Exported 3D surface plot to exports/bci_focus_analysis.html")